In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split



In [5]:
df = pd.read_csv('iris.csv')
df = df.dropna().reset_index(drop=True)

# df = df.drop(columns=['Species'])

target_col = df.columns[-1]
X_df = df.drop(columns=[target_col]).copy()
y_raw = df[target_col].values

le = LabelEncoder()
y = le.fit_transform(y_raw)

X_new_df = pd.DataFrame([[6, 3, 0.7, 0.9]], columns = X_df.columns)

for col in X_new_df.columns:
    if X_df[col].dtype == 'object':
        enc = LabelEncoder()
        X_df[col] = enc.fit_transform(X_df[col])
        X_new_df[col] = enc.transform(X_new_df[col])

X_raw = X_df.values.astype(float)
X_new_raw = X_new_df.values.astype(float)

a,b = 0, 1

scaler = MinMaxScaler(feature_range=(a,b))
X = scaler.fit_transform(X_raw)
X_new = scaler.transform(X_new_raw)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

print(f"So mau: {len(X)}, So cot: {X.shape[1]}, Cac lop: {le.classes_}")


So mau: 150, So cot: 4, Cac lop: ['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']


A. Naive Bayes

In [7]:
def gaussian(x, mean, var):
    eps = 1e-9
    return (1.0 / np.sqrt(2.0 * np.pi * ( var + eps))) * np.exp(-((x - mean) ** 2 / ( 2 + ( var + eps ))))

classes = np.unique(y)
probs = {}

for c in classes:
    X_c = X[y == c]
    prior = len(X_c) / len(X)
    mean, var = X_c.mean(axis = 0 ), X_c.var(axis = 0)
    likelihood = np.prod( gaussian(X_new[0], mean, var ))
    probs[c] = prior * likelihood

total = sum(probs.values()) + 1e-15
for c in classes:
    probs[c] /= total
    print(f"P(X thuoc {le.inverse_transform([c])[0]}) = {probs[c]: .4f}")

pred_c = max(probs, key = probs.get)
print("X thuoc lop: ", le.inverse_transform([pred_c])[0])



P(X thuoc Iris-setosa) =  0.8294
P(X thuoc Iris-versicolor) =  0.1242
P(X thuoc Iris-virginica) =  0.0465
X thuoc lop:  Iris-setosa


In [8]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(X, y)
print("Xac suat: ", np.round(nb.predict_proba(X_new)[0], 4))
print("X thuoc lop: ", le.inverse_transform(nb.predict(X_new))[0])


Xac suat:  [0.5157 0.4843 0.    ]
X thuoc lop:  Iris-setosa


B. KMeans

In [9]:
K = 2
np.random.seed(42)

centroids = X[np.random.choice(len(X), K, replace = False)]

for _ in range(100):
    distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis = 2)
    labels = np.argmin(distances, axis = 1)

    new_centroids = np.array([X[labels == k].mean(axis = 0) if np.any(labels == k) else centroids[k] for k in range(K)])

    if np.allclose(centroids, new_centroids):
        break
    centroids = new_centroids

for k in range(K):
    print(f"Cum {k}: {np.sum( labels == k)} man | Tam: {np.round(centroids[k], 3)}")

print("X_new thuoc cum: ", np.argmin(np.linalg.norm(centroids - X_new[0], axis = 1)))


Cum 0: 100 man | Tam: [0.545 0.363 0.662 0.657]
Cum 1: 50 man | Tam: [0.196 0.591 0.079 0.06 ]
X_new thuoc cum:  1


In [10]:
from sklearn.cluster import KMeans

km = KMeans( n_clusters = K, random_state = 42, n_init = 10)
labels = km.fit_predict(X)

for k in range(K):
    print(f"Cum {k}: {np.sum(labels == k)} mau | Tam: {np.round(km.cluster_centers_[k], 3)}")

print("X_new thuoc cum: ", km.predict(X_new)[0])

Cum 0: 100 mau | Tam: [0.545 0.363 0.662 0.657]
Cum 1: 50 mau | Tam: [0.196 0.591 0.079 0.06 ]
X_new thuoc cum:  1


C. Phan cum phan cup - Thuat toan gop

In [11]:
K = 3

clusters = [[i] for i in range(len(X))]
centers = [X[i] for i in range(len(X))]

while len(clusters) > K:
    min_dist, pair = float('inf'), (0,1)
    for i in range(len( centers )):
        for j in range(i+1, len(centers)):
            d = np.linalg.norm(centers[i] - centers[j])
            if d < min_dist:
                min_dist, pair = d, (i, j)

    i, j = pair
    clusters[i].extend(clusters[j])
    centers[i] = X[clusters[i]].mean( axis = 0 )
    clusters.pop(j)
    centers.pop(j)

for idx, c in enumerate(clusters):
    print(f"Cum {idx}: {len(c)} mau")

print("X_new thuoc cum:", np.argmin([np.linalg.norm(c - X_new[0]) for c in centers]))


Cum 0: 49 mau
Cum 1: 1 mau
Cum 2: 100 mau
X_new thuoc cum: 0


In [12]:
from sklearn.cluster import AgglomerativeClustering

agg = AgglomerativeClustering(n_clusters = K, linkage = 'ward')
labels_agg = agg.fit_predict(X)

centers_lib = []
for k in range(K):
    print(f"Cum {k}: {np.sum(labels_agg == k)} mau")
    centers_lib.append(X[labels_agg == k].mean(axis=0))

print(f"X_new thuoc cum:", np.argmin([np.linalg.norm(c - X_new[0]) for c in centers_lib]))

Cum 0: 67 mau
Cum 1: 50 mau
Cum 2: 33 mau
X_new thuoc cum: 1


D. Decision Tree/ ID3

In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

clf = DecisionTreeClassifier( criterion='entropy', random_state = 42)
clf.fit(X_train, y_train)
print("Accury:", round(accuracy_score(y_test, clf.predict(X_test)), 4))
print("Du doan X_new:", le.inverse_transform(clf.predict(X_new))[0])


Accury: 1.0
Du doan X_new: Iris-setosa


E. Mang NoRon

In [15]:
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier

# Perceptron 1 lop
ppn = Perceptron(max_iter = 200, eta0 = 0.01, random_state = 42)
ppn.fit(X_train, y_train)
print("Accuracy:", round(ppn.score(X_test, y_test), 4), "X_new:", le.inverse_transform(ppn.predict(X_new))[0])

#MLP nhieu lop
mlp = MLPClassifier(hidden_layer_sizes = (16,8), activation = 'relu', max_iter = 2000, random_state = 42)
mlp.fit(X_train, y_train)
print( "Accuracy", round(mlp.score(X_test, y_test), 4),
       "X_new:", le.inverse_transform(mlp.predict(X_new))[0])


Accuracy: 0.8 X_new: Iris-setosa
Accuracy 0.9667 X_new: Iris-setosa


F. Apriori & FPGrowth

In [ ]:
# df_basket = pd.read_csv('basket.csv', header = None)

transactions = [set(str(item) for item in row if pd.notna(item)) for row in df_basket.values]
transactions = [set(row) for row in sample_baskets]

sample_baskets = [
    ['milk', 'bread', 'eggs'],
    ['milk', 'bread'],
    ['milk', 'eggs'],
    ['bread', 'eggs'],
    ['milk', 'bread', 'eggs']
]

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

te = TransactionEncoder()
df_enc = pd.DataFrame(te.fit_transform([list(t) for t in transactions]), columns = te.columns_)

freq_items = fpgrowth(df_enc, min_support=0.25, use_colnames=True)

rules = association_rules(freq_items, metric = 'confidence', min_threshold=0.8)
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())
